# Module 1 – Assignment: Build a Simple ETL Pipeline

## Instructions
- Run the cells **in order** from top to bottom.
- Write your code in the cells marked `# Your code here`.
- Do **not** change the variable names given to you — they are used in later steps.
- When you finish, save the notebook and submit it.

---

## Scenario

You are a data engineer at a **library**. The library tracks **book loans** every month.  
You will build a small ETL pipeline that:
1. **Extracts** raw loan records from a CSV file  
2. **Transforms** the data (clean it and add new columns)  
3. **Loads** the clean data into a SQLite database

---


## Step 0 – Setup

Run this cell first. It imports all the libraries you will need.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os

print("Libraries loaded!")


Libraries loaded!


---

## Step 1 – Generate the Raw Data

The cell below creates the raw data for you and saves it as `library_loans.csv`.  
**Read it carefully so you understand the data, then run it.**


In [2]:
# ── This cell creates your data – just run it, do not change it ──
np.random.seed(10)
n = 200

raw = pd.DataFrame({
    'loan_id'     : range(1, n + 1),
    'member_id'   : np.random.randint(200, 260, n),
    'book_id'     : np.random.choice(['B01','B02','B03','B04','B05',
                                       'B06','B07','B08','B09','B10'], n),
    'loan_date'   : pd.date_range('2024-09-01', periods=n, freq='4h')
                      .strftime('%Y-%m-%d'),
    'return_date' : pd.date_range('2024-09-15', periods=n, freq='4h')
                      .strftime('%Y-%m-%d'),
    'genre'       : np.random.choice(['Fiction','Science','History',
                                       'Technology','Art', None], n,
                                      p=[0.3,0.2,0.2,0.15,0.1,0.05]),
    'late_fee'    : np.random.choice([0.0, 0.0, 0.5, 1.0, 2.0, None], n,
                                      p=[0.5,0.2,0.1,0.1,0.05,0.05])
})

# Inject a few problems
dup = raw.sample(10, random_state=3)
raw = pd.concat([raw, dup], ignore_index=True)           # 10 duplicate rows
raw.loc[np.random.choice(raw.index, 6, replace=False), 'late_fee'] = -1.0  # 6 invalid fees

raw.to_csv('library_loans.csv', index=False)
print(f"library_loans.csv created: {len(raw)} rows")
print(f"Columns: {list(raw.columns)}")


library_loans.csv created: 210 rows
Columns: ['loan_id', 'member_id', 'book_id', 'loan_date', 'return_date', 'genre', 'late_fee']


---

## Step 2 – EXTRACT

Read the CSV file you just created into a DataFrame called `df`.  
Then print its shape (number of rows and columns).


In [3]:
# TASK 2 – Read library_loans.csv into df
# Your code here:
df = pd.read_csv("library_loans.csv")

# Print shape
print("Shape:", df.shape)


Shape: (210, 7)


---

## Step 3 – Explore the Data

Run a quick quality check before you clean anything.  
Print:
- The first 5 rows  
- The count of **duplicate rows**  
- The count of **null values** in each column  
- The count of rows where `late_fee` is **negative**


In [4]:
# TASK 3 – Data quality check
# Your code here:
df.head(5)
print(f"Duplicated Rows: {df.duplicated().sum()}")
print("----------------")
print("Null Value Counts: ")
print(df.isnull().sum())
print("----------------")
print(f"Negative Value: {(df['late_fee'] < 0).sum()}")




Duplicated Rows: 9
----------------
Null Value Counts: 
loan_id        0
member_id      0
book_id        0
loan_date      0
return_date    0
genre          9
late_fee       8
dtype: int64
----------------
Negative Value: 6


---

## Step 4 – TRANSFORM

Complete **all four** cleaning steps below.

### 4a – Remove duplicate rows


In [5]:
# TASK 4a – Drop duplicate rows and save the result back into df
# Your code here:
df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 201


### 4b – Fill missing values

Fill missing `genre` values with `'Unknown'` and missing `late_fee` values with `0.0`.

In [6]:
# TASK 4b – Fill nulls
# Your code here:
df['genre'] = df['genre'].fillna('Unkown')
df['late_fee'] = df['late_fee'].fillna(0.0)
print("Nulls remaining:", df[['genre','late_fee']].isnull().sum().to_dict())


Nulls remaining: {'genre': 0, 'late_fee': 0}


### 4c – Remove invalid records

Remove any rows where `late_fee` is **less than 0**.

In [7]:
# TASK 4c – Remove rows with negative late_fee
# Your code here:


df= df[df['late_fee'] >= 0]
print("Rows after removing invalid fees:", len(df))


Rows after removing invalid fees: 195


### 4d – Add a new column

Create a new column called `loan_days` that shows how many days each loan lasted.

**Hint:** Convert `loan_date` and `return_date` to datetime, then subtract them.

In [8]:
# TASK 4d – Add loan_days column
# Your code here:
df['loan_date'] = pd.to_datetime(df['loan_date'])
df['return_date'] = pd.to_datetime(df['return_date'])
df['loan_days'] = (df['return_date'] - df['loan_date']).dt.days
print(df[['loan_id', 'loan_date', 'return_date', 'loan_days']].head())


   loan_id  loan_date return_date  loan_days
0        1 2024-09-01  2024-09-15         14
1        2 2024-09-01  2024-09-15         14
2        3 2024-09-01  2024-09-15         14
3        4 2024-09-01  2024-09-15         14
4        5 2024-09-01  2024-09-15         14


---

## Step 5 – LOAD

Load the clean DataFrame into a SQLite database called `library.db`, in a table named `clean_loans`.  
Then verify by querying the table and printing the row count.


In [9]:
# TASK 5 – Load to SQLite and verify
# Your code here:
conn = sqlite3.connect('library.db')
df.to_sql('clean_loans', conn, if_exists='replace', index=False)
# Verify

result = pd.read_sql("SELECT COUNT(*) as total_rows FROM clean_loans", conn)
print("Rows in database:", result['total_rows'][0])
conn.close() 


Rows in database: 195


---

## Step 6 – Simple Analysis

Answer this question using your clean data:  
**Which genre has the highest total late fees?**

Print the result as a small table with two columns: `genre` and `total_late_fee`.


In [10]:
# BONUS – Genre with highest total late fees
# Your code here:
df.groupby(['genre']).agg(
    total_late_fee = ('late_fee','sum')
).sort_values(by='total_late_fee', ascending=False)


,total_late_fee
genre,
Fiction,16.5
Technology,12.0
Science,9.0
History,9.0
Art,4.0
Unkown,0.5



---
*Module 1 Assignment – Practical Data Engineering*
